<a href="https://colab.research.google.com/github/Mxrtmlk/Talleres-tica/blob/main/TallerSumativo_2_Casen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1> ETI195 - Ética para Ciencia de Datos y Estadística </h1><center>
<center><h2> Taller Sumativo II: Sesgos, justicia, y discriminación <h2></center>

Vicente Rodríguez, Martín Pérez, Fernando Pavez

# `Instrucciones Generales`


## `Temas formales`

**Modalidad**: Grupos de 3 personas.

**Formato de entrega:** El trabajo debe entregarse a través de Canvas, tanto en formato Jupyter ejecutable (.ipynb) como en formato PDF.

**Formalidades de las respuestas:**
- Citas: APA 7.
- Se espera buena ortografía y redacción, con respuestas en tono académico.

## `Recomendaciones`

- Asistan a los talleres formativos, revisen los ejemplos de código y consulten sus dudas con sus ayudantes.

- **Lean la rúbrica antes de hacer su tarea.**

## `Integridad académica y uso de IA`

El uso de asistentes de IA sigue los lineamientos de integridad académica del curso, con las siguientes excepciones:

- Pueden usar IA para consultar por bugs y entender mejor el código.
- Pueden usar IA para generar las representaciones visuales que estimen convenientes.

Si tienen dudas sobre el uso de IA o la citación de código externo, no duden en preguntar. Pueden usar libremente los códigos de los talleres formativos. <3

---
## `Contexto`

La Encuesta de Caracterización Socioeconómica Nacional [CASEN 2024](https://observatorio.ministeriodesarrollosocial.gob.cl/encuesta-casen) cumple un rol fundamental en la toma de decisiones para la formulación de políticas sociales en Chile. Los organismos gubernamentales buscan estimar la realidad de la población, por lo que cualquier modelo predictivo construido a partir de estos datos tiene la capacidad de influir en la asignación de recursos y en la formulación de políticas públicas.

En ese sentido, un modelo que prediga la situación socioeconómica de la población puede transformarse en una herramienta con gran poder y grandes riegos.

Como científicos de datos, a menudo nos enfrentamos a este tipo de situaciones mediante dos preguntas: ¿cómo resolver automáticamente un problema de clasificación? Y, quizás más importante, ¿qué puede salir mal cuando ese modelo toma decisiones que afectan a personas?

En este taller entrenarán un modelo predictivo de pobreza multidimensional (`pobreza_multi`) utilizando el [dataset preprocesado](https://drive.google.com/file/d/15NMlO4BD-OB55uvfYWUSCh_ba4-zCzI8/view?usp=drive_link) de la encuesta [CASEN 2024](https://observatorio.ministeriodesarrollosocial.gob.cl/encuesta-casen).

El objetivo es detectar posibles sesgos en las predicciones, evaluar si el modelo es justo para distintos grupos de la población, aplicar técnicas de mitigación y discutir los riesgos de automatizar este tipo de problemas de clasificación.

---

## Parte 1

Lea la [**Metodología de Diseño Muestral CASEN 2024**](https://drive.google.com/file/d/1VafJ57Ta6_gwUM-8sir6PAw5DVuBh0rq/view?usp=sharing), específicamente la sección *Antecedentes y características generales del diseño*, y revise el [**libro de códigos de la base de datos CASEN 2024**](https://docs.google.com/spreadsheets/d/1x25URkqJy7Ncc61VEc75NE7XKFsrUHw5/edit?gid=937549388#gid=937549388) para entender las variables disponibles.

Antes de escribir una sola línea de código, lean los materiales de contexto mencionados anteriormente. Deben comprender por qué se levanta la encuesta CASEN, qué mide la variable `pobreza_multi`

### **RESPONDER**

1.  Discuta los objetivos de la encuenstra CASEN con referencias a la metodología y  para qué serviría entrenar  un clasificador automático de este tipo.

Según el Ministerio de Desarrollo Social y Familia (MDSF, 2026, p. 10), la Encuesta de Encuesta de Caracterización Socioeconómica Nacional (CASEN) es el principal instrumento de diagnóstico y evaluación del Estado chileno, cuyo propósito central es ver periódicamente las condiciones de vida de los hogares y de la población, con un énfasis en los grupos prioritarios y en situación de vulnerabilidad. Este instrumento recopila información multidimensional que abarca aspectos demográficos, educativos, de salud, vivienda, trabajo e ingresos, permitiendo además estimar la cobertura, la focalización y la distribución del gasto fiscal de los distintos programas sociales implementados a nivel nacional.

Al revisar la base de datos y su libro de códigos, la variable pobreza_multi es el indicador de pobreza multidimensional, el cual clasifica a los hogares en dos categorías (0 para hogares fuera de pobreza multidimensional y 1 para hogares en esta situación). A diferencia de la medición tradicional basada estrictamente en los ingresos monetarios, la pobreza multidimensional evalúa de manera integral las carencias simultáneas que experimentan los hogares en múltiples ámbitos esenciales para el bienestar y la dignidad humana.

En este contexto, entrenar un clasificador automático que pueda predecir la situación de pobreza multidimensional nos da la ventaja de automatizar la identificación de vulnerabilidades socioeconómicas a gran escala. Sin embargo, el uso de estos modelos predictivos en el diseño de políticas públicas tiene varios riesgos éticos y sociales significativos. Tratándose de datos sensibles que determinan la asignación de recursos o el otorgamiento de beneficios estatales, un modelo automatizado posee un poder considerable, pero también el riesgo de propagar sesgos algorítmicos. Esto podría derivar en la exclusión errónea de sectores de la población que realmente necesitan el apoyo estatal, impidiendo su acceso a subsidios esenciales para su subsistencia.

Una problemática similar se ha visto con el sistema de clasificación SISFOH en Perú, donde las fallas en los modelos predictivos dejaron temporalmente sin beneficios sociales a numerosos adultos mayores en situación de vulnerabilidad. Por esto mismo, el desarrollo de herramientas predictivas automatizadas en este ámbito socioeconómico tiene que ser hecho con rigurosidad, transparencia y una revisión constante de equidad para evitar producir o profundizar desigualdades estructurales.

## Librerias y configuraciones

In [1]:
# imports iniciales
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# sklearn - importa aquí las librerías necesarias
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

# aequitas
from aequitas.group import Group
from aequitas.bias import Bias

# aif360
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.postprocessing.eq_odds_postprocessing import EqOddsPostprocessing
from aif360.algorithms.preprocessing import Reweighing

import warnings
warnings.filterwarnings("ignore")

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


## Parte 2: Realice el preprocesamiento de los datos y entrene un árbol de decisión para predecir el nivel de pobreza multidimensional (pobreza_multi). Reporte el accuracy y la matriz de confusión sobre los datos de testeo.

*Puede reutilizar el jupyter o trabajar con el mismo jupyter de la entrega anterior, puede entregar una carpeta con ambos jupyter y funciones compartidas*

### Indicaciones

- El preprocesamiento debe incluir al menos: manejo de valores faltantes, codificación de variables categóricas, normalización de variables númericas y tratamiento de valores extremos

- Utilice random_state o semillas donde corresponda para garantizar la reproducibilidad de los resultados.

- La clase pobreza_multi está desbalanceada. Investigue el argumento class_weight='balanced' de sklearn para manejar este problema y aplíquelo si lo considera pertinente.


### **RESPONDER**

2.1 Explique y justifique los pasos del preprocesamiento.

2.2 ¿Podemos concluír con esto que el modelo tiene el mismo rendimiento para todos los grupos demográficos?


In [2]:
# Cargamos el dataset

file_id = "1DQsXd6hThmoROcqSqVlBPEO-jb0joKj9"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

casen = pd.read_csv(url)

# Dimensiones de la matriz de información
print(casen.shape)
print(f"El dataset tiene {casen.shape[0]} filas y {casen.shape[1]} columnas")

(212471, 42)
El dataset tiene 212471 filas y 42 columnas


En la siguiente celda realizamos la conversión de las columnas que, aunque venían cargadas con formato numérico en la base de datos, en realidad corresponden a variables cualitativas (códigos de respuesta, escalas u opciones discretas). Esto tiene como objetivo asegurar que el tipo de dato sea el correcto, facilitando que más adelante puedan ser procesadas adecuadamente como variables explicativas X en el clasificador.

In [3]:
# Listado de variables que son numéricas pero corresponden a categorías
vars_to_cat = ['p2', 'p3', 'p4', 'p9', 'h7e', 'h7f', 'r14', 'r17a', 'r17b', 'r17c', 'v3', 'v5', 'v23']

# Cambiamos el tipo
for col in vars_to_cat:
    if col in casen.columns:
        casen[col] = casen[col].astype('category')

# Verificamos los cambios
print(casen[vars_to_cat].dtypes)

p2      category
p3      category
p4      category
p9      category
h7e     category
h7f     category
r14     category
r17a    category
r17b    category
r17c    category
v3      category
v5      category
v23     category
dtype: object


A continuación, cuantificamos la cantidad de valores nulos presentes por columna en todo el dataset. El objetivo de este paso es comprobar el estado de integridad de los datos, al confirmar que no existen valores faltantes, confirmamos que no hay necesidad de aplicar técnicas de imputación (como relleno) o eliminación de registros por este motivo.

In [4]:
# Verificamos la cantidad de nulos por columna
print(casen.isnull().sum())

area             0
p1               0
p2               0
p3               0
p4               0
p9               0
tot_per_h        0
edad             0
sexo             0
ind_hacina       0
e1               0
e9depen          0
o1               0
o19              0
o31              0
s13              0
h7e              0
h7f              0
r1a              0
r3               0
r14              0
r17a             0
r17b             0
r17c             0
v1               0
v2               0
v3               0
v4               0
v5               0
v6               0
v13              0
v20              0
v22              0
v23              0
v24              0
v24b             0
v27a             0
v27b             0
v29b             0
os1              0
trans            0
pobreza_multi    0
dtype: int64


In [5]:
# Definimos la matriz de características (X) y variable objetivo (y)
X = casen.drop(columns=['pobreza_multi'])
# Convertimos la variable objetivo a formato binario (1 para Pobre y 0 para No pobre)
y = casen['pobreza_multi'].apply(lambda x: 1 if str(x).strip().lower() == 'pobre' else 0)

In [6]:
# Identificamos columnas numéricas y categóricas
n_columnas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
c_columnas = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [7]:
# Construimos los transformadores
# Ambos tienen un imputer en el caso de nulos (aunque no los tengamos los ponemos para tener un doble chequeo
# y en el caso que hubieran que se corrijan sin afectar de  gran manera a los datos)
# En el caso de las numéricas tienen un escalado para evitar un peso mayor a variables
# solo por estar en distintos rangos y para las categorías se les asigna un valor binario a cada categoría
# Finalmente unimos todo en el preprocesador que estará listo para entregarse al modelo.

transformador_n = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

transformador_c = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())])

preprocesador = ColumnTransformer(
    transformers=[
        ('num', transformador_n, n_columnas),
        ('cat', transformador_c, c_columnas)])

In [8]:
# Dividimos los datos en entrenamiento y testeo (con una seed fija)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
# Calculamos los porcentajes exactos de la variable objetivo pobreza_multi
conteo_clases = casen['pobreza_multi'].value_counts()
porcentajes_clases = casen['pobreza_multi'].value_counts(normalize=True) * 100

print("\nPorcentajes:")
print(porcentajes_clases.round(2))


Porcentajes:
pobreza_multi
No pobre    82.35
Pobre       17.65
Name: proportion, dtype: float64


Viendo esto, podemos notar que las clases están notoriamente desbalanceadas, ya que el 82.4% de los datos corresponden a hogares "No pobre" frente a solo un 17.6% clasificados en la categoría de "Pobre". Dado que los datos están distribuidos de esta forma en pobreza_multi, para mitigar este problema vamos a usar el argumento class_weight='balanced', lo que nos servirá para evitar el sesgo hacia la clase mayoritaria y no penalizar los errores en la clase minoritaria, esto es especialmente importante en el contexto de datos sensibles, ya que nos ayuda a minimizar los errores de detección de poblaciones vulnerables (como los falsos negativos) para evitar excluir a quienes realmente necesitan el apoyo estatal.

In [10]:
# Definimos el modelo de Árbol de Decisión considerando el desbalance de clases
clasificador = DecisionTreeClassifier(
    criterion='gini',
    max_depth=10,
    class_weight='balanced',
    random_state=42)

In [11]:
# Unimos preprocesamiento y modelo en un Pipeline y lo entrenamos
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocesador),
    ('classifier', clasificador)])

model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tot_per_h', 'edad',
                                                   'ind_hacina', 'o1', 'v24b',
                                                   'v27a', 'v27b', 'v29b']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['area', 'p1', 'p2', 'p3',
                                                   'p4', 'p9', 'sexo', 'e1',
                                                   'e9depen', 'o19', 'o31',
                                                   's13', 'h7e', 'h7f', 'r1a',
                                                   'r3', 'r14', 'r17a', 'r17b',
                                                   'r17c', 'v1', 'v2', 'v3',
                                                   'v4', 'v5', 'v6', 'v13',
                                                   'v20', 'v22', 'v23', ...])])),
                ('classifier',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                                        random_state=42))])

In [12]:
# Finalmente evaluamos el rendimiento en el conjunto de testeo
prediccion_y = model_pipeline.predict(X_test)

acc = accuracy_score(y_test, prediccion_y)
cm = confusion_matrix(y_test, prediccion_y)

print(f"Accuracy al testear: {acc:.5f}")
print("Matriz de Confusión:\n", cm)

Accuracy al testear: 0.74239
Matriz de Confusión:
 [[25765  9231]
 [ 1716  5783]]


En respuesta a la pregunta de la sección *2.2*, no es posible concluir que el modelo tenga el mismo rendimiento para todos los grupos demográficos, ya que a pesar de haber alcanzado un accuracy global de 0.74239 y una matriz de confusión general en el conjunto de testeo, estas métricas solo reflejan el comportamiento agregado del árbol de decisión sin desglosar el desempeño por subpoblaciones. Aunque se haya mitigado el desbalance de clases mediante el parámetro de peso balanceado, las métricas globales son insuficientes para descartar sesgos ocultos, por lo que resulta indispensable realizar una revisión de equidad específica sobre las distintas variables sensibles y demográficas para asegurar un rendimiento verdaderamente justo y transversal.

## Parte 3: Evalúe las predicciones del modelo mediante las métricas de Fairness de Aequitas.

### **Indicaciones**

Considere como posibles atributos demográficos sensibles: genero, r3 (pertenencia a pueblo indígena), os1 (orientación sexual) y edad_cat; o bien atributos asociados a posibles discapacidades (revise el libro de códigos): h7e y h7f.

**Paso 1: Binarice las variables sensibles.**
Aequitas requiere que todas las variables sensibles estén binarizadas en grupos privilegiado / no privilegiado. Para variables con más de dos categorías, cree una columna auxiliar que agrupe las categorías y justifique su decisión.

**Paso 2: Elija al menos 2 atributos sensibles y defina sus grupos.**

Ejemplos de posibles agrupaciones, y grupos considere que puede elegir sus propias categorizaciones de privilegio y los grupos que estime.:

| Atributo | Grupo | Categoría |
|---|---|---|
| r3 | Privilegiado | no indígena |
| r3 | No privilegiado | indígena |
| os1 | Privilegiado | heterosexual |
| os1 | No privilegiado | otras orientaciones |
| edad_cat | Privilegiado | *(a definir según los intervalos que usted cree, justificando la elección)* |
| edad_cat | No privilegiado | *(a definir según los intervalos que usted cree, justificando la elección)* |

o, asociados a discapacidad:

| Atributo | Grupo | Categoría |
|---|---|---|
| h7e | Privilegiado | 1. No, ninguna dificultad |
| h7e | No privilegiado | las demás categorías |
| h7f | Privilegiado | 1. No, ninguna dificultad |
| h7f | No privilegiado | las demás categorías |

Pueden

**Paso 3: Reporte las métricas.**
Reporte las métricas de *False Negative Rate Parity* y *False Discovery Rate Parity* para los atributos y grupos definidos.

### **RESPONDER**

Pregunta 3.1 Explique los resultados obtenidos.

Pregunta 3.2 ¿Cuán justas se pueden considerar las predicciones del modelo para cada grupo? ¿Cómo relacionaría esto con los contenidos del curso?

In [13]:
### codigo aquí
#veo en q estan separados los valores de cada variable para desps categorizarloes en privilegiados o no
# en este caso usaremos los valores de si pertenecen al pueblo indigena o si pueden aseasrse o vestirse solos
print(casen["h7e"].unique())
print(casen["h7e"].value_counts())
print(casen["r3"].unique())
print(casen["r3"].value_counts())
print(casen["sexo"].unique())
print(casen["sexo"].value_counts())

[1, 4, 2, 3]
Categories (4, int64): [1, 2, 3, 4]
h7e
1    204192
2      4611
3      2045
4      1623
Name: count, dtype: int64
['No indígena' 'Indígena']
r3
No indígena    182826
Indígena        29645
Name: count, dtype: int64
['Hombre' 'Mujer']
sexo
Mujer     111619
Hombre    100852
Name: count, dtype: int64


In [14]:
df_test = X_test.copy()

df_test["label_value"] = y_test.values
df_test["score"] = prediccion_y

Binarizacion de todas las variables sensibles, luego elegiremos r3 y h7e para entrenar el modelo

In [15]:
#para r3 consideramos privilegiados a las personas que no pertenecen a pueblos indigenas
df_test["r3_bin"] = df_test["r3"].apply(
    lambda x: "privilegiado" if x == "No indígena"
    else "no_privilegiado"
)
#para h73 consideramos privilegiados solo a quienes pueden asearse o vestirse sin dificultad alguna
df_test["h7e_bin"] = df_test["h7e"].apply(
    lambda x: "privilegiado" if x == 1 else "no_privilegiado"
)
# tomamos hombre = 1 como privilegiado
df_test["sexo_bin"] = df_test["sexo"].apply(
    lambda x: "privilegiado" if x == 1 else "no_privilegiado"
)
# sin dificultad para comunicarse es privilegiado
df_test["h7f_bin"] = df_test["h7f"].apply(
    lambda x: "privilegiado" if x == 1 else "no_privilegiado"
)
# en cuanto al hacinamiento sin hacinamiento es priv
df_test["ind_hacina_bin"] = df_test["ind_hacina"].apply(
    lambda x: "privilegiado" if x == 1 else "no_privilegiado"
)
#consideramos a los heterosexuales como privilegiados
df_test["os1_bin"] = df_test["os1"].apply(
    lambda x: "privilegiado" if x == 1 else "no_privilegiado"
)
#En cuanto a la edad consideramos privilegiados a las personas mayores de edad pero que no sean de la 3era edad
df_test["edad_bin"] = df_test["edad"].apply(
    lambda x: "no_privilegiado" if x < 18 or x > 60
    else "privilegiado"
)

In [16]:
df_aequitas = df_test[[
    "r3_bin",
    "h7e_bin",
    "label_value",
    "score"
]].copy()

print(df_aequitas.head())

                 r3_bin       h7e_bin  label_value  score
97370      privilegiado  privilegiado            0      0
158738     privilegiado  privilegiado            0      0
208639     privilegiado  privilegiado            0      0
200894  no_privilegiado  privilegiado            0      1
9934       privilegiado  privilegiado            1      1


In [17]:
#Buscamos el FNR Y FDR
g = Group()

group_metrics, _ = g.get_crosstabs(
    df_aequitas,
    attr_cols=["r3_bin", "h7e_bin"]
)

In [18]:
#metricas por grupo
metricas = group_metrics[[
    "attribute_name",
    "attribute_value",
    "fnr",
    "fdr"
]]

display(metricas)

,attribute_name,attribute_value,fnr,fdr
0,r3_bin,no_privilegiado,0.197566,0.577619
1,r3_bin,privilegiado,0.235988,0.622816
2,h7e_bin,no_privilegiado,0.286041,0.560563
3,h7e_bin,privilegiado,0.225290,0.617520


In [19]:
# metricas q muestran la diff respecto al grupo privilegiado
b = Bias()

bias_metrics = b.get_disparity_predefined_groups(
    group_metrics,
    original_df=df_aequitas,
    ref_groups_dict={
        "r3_bin": "privilegiado",
        "h7e_bin": "privilegiado"
    }
)

display(bias_metrics)

,model_id,score_threshold,k,attribute_name,attribute_value,accuracy,tpr,tnr,for,fdr,...,ppr_ref_group_value,pprev_ref_group_value,precision_ref_group_value,fdr_ref_group_value,for_ref_group_value,fpr_ref_group_value,fnr_ref_group_value,tpr_ref_group_value,tnr_ref_group_value,npv_ref_group_value
0,0,binary 0/1,15014,r3_bin,no_privilegiado,0.696934,0.802434,0.664698,0.083258,0.577619,...,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado
1,0,binary 0/1,15014,r3_bin,privilegiado,0.749822,0.764012,0.746976,0.059588,0.622816,...,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado
2,0,binary 0/1,15014,h7e_bin,no_privilegiado,0.682067,0.713959,0.670530,0.133690,0.560563,...,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado
3,0,binary 0/1,15014,h7e_bin,privilegiado,0.744823,0.774710,0.738576,0.059934,0.617520,...,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado,privilegiado


In [20]:
print(bias_metrics.columns.tolist())

['model_id', 'score_threshold', 'k', 'attribute_name', 'attribute_value', 'accuracy', 'tpr', 'tnr', 'for', 'fdr', 'fpr', 'fnr', 'npv', 'precision', 'pp', 'pn', 'ppr', 'pprev', 'fp', 'fn', 'tn', 'tp', 'group_label_pos', 'group_label_neg', 'group_size', 'total_entities', 'prev', 'ppr_disparity', 'pprev_disparity', 'precision_disparity', 'fdr_disparity', 'for_disparity', 'fpr_disparity', 'fnr_disparity', 'tpr_disparity', 'tnr_disparity', 'npv_disparity', 'ppr_ref_group_value', 'pprev_ref_group_value', 'precision_ref_group_value', 'fdr_ref_group_value', 'for_ref_group_value', 'fpr_ref_group_value', 'fnr_ref_group_value', 'tpr_ref_group_value', 'tnr_ref_group_value', 'npv_ref_group_value']


In [21]:
display(
    bias_metrics[
        [
            "attribute_name",
            "attribute_value",
            "fdr_disparity",
            "fnr_disparity"
        ]
    ]
)

,attribute_name,attribute_value,fdr_disparity,fnr_disparity
0,r3_bin,no_privilegiado,0.927431,0.837187
1,r3_bin,privilegiado,1.000000,1.000000
2,h7e_bin,no_privilegiado,0.907766,1.269656
3,h7e_bin,privilegiado,1.000000,1.000000


Para r3(caso indigena o no), el grupo no privilegiado (indígena) presenta un FDR disparity de 0,927 y un FNR disparity de 0,837 respecto al grupo privilegiado (no indígena). Dado que los dos son menores que 1, por lo que las tasas de falsos descubrimientos y falsos negativos son menores para el grupo no privilegiado que para el grupo privilegiado. Aplicado al caso de casen podemos ver q deberia existir una diferencia economica entre estos 2 grupos y tal vez no se este mostrando de la froma mas correcta posible, pero no necesariamente se lo podriamos atribuir al modelo.

En cuanto a h7e podemos ver q la tasa de falsos negativos es bastante grande al ser 1.27 aprox, esto en casen podria determinar q alguien que si puede pertenecer a un grupo mas vulnerable quede clasificado como no vulnerable.


3.1 : Tal como acabo de mencionar en r3 podemos ver q hay una tasa un poco menor a 1 de fdr por lo que si bien , podria mejorarse esa tasa tal vez mejorando el modelo o agregando mas parametros para responder en torno a los pueblos indigenas para mejorarlo, pero no es tan preocupante la disparidad en este caso. Por otra parte la fnr ya podria darnos una señal para tal vez observarlo de cerca pero no para alarmarse, ya que empieza a aumentar la brecha respecto al grupo de control (privilegiados).
Donde si hay que mostrar una preocupacion es en el fnr disparity de h7e, ya que la disparidad comienza a ser bastante mayor en este caso para personas con dificultades en su cuidado personal, las cuales son clasificadas erroneamente por el sistema(27% mayor q el grupo privilegiado).Sin embaro nuevamente no podemos decir con certeza que es culpa directamente del modelo o no al tener tantas variables involucradas.


3.2 Si basamos lo que es "Justo" solo en nuestras relaciones de fdr y fnr, junto con sus disparidades respecto al grupo privilegiado, y que consideremos justo que todas las personas respecto a todos sus contextos se les clasifique correctamente su situacion de vulnerabilidad, podriamos decir que para ambas categorias (indigenas y cuidado personal) sus fdr son mas o menos justos, ya que no se distancian en exceso del grupo general. Sin embargo en los fnr de estas 2 categorias deberiamos centrarnos mucho mas , ya que son numeros que estan al borde (indigenas) o directamente preocupantes(capacidad de aseo personal).Sin embargo no podemos decir directamente que el programa es injusto o si tiene algun problema en especifico debido a la enorme cantidad de factores que aplican para determinar la vulnerabilidad de cada individuo. Podriamos intentar de mejorar esta cantidad de falsos negativos respecto a los falsos negativos generales, para eso tendriamos que investigar en profundidad estos casos de falsos negativos, para determinar si existe un problema directo en como el programa distribuye el aumento de probabilidad de vulneracion a una persona respecto a estas variables, si existe tal vez un problema en las preguntas o si faltan mas preguntas o mas grados de clasificacion,etc.

## Parte 4: Aplique el algoritmo EqOddsPostProcessing sobre las predicciones del modelo de la pregunta 1. Obtenga las predicciones modificadas y evalúelas.

#### Indicaciones:

- Obtenga tanto las instancias de BinaryLabelDatasetMetric como de ClassificationMetric y calcule al menos una métrica de Fairness.

### **RESPONDER**

pregunta 4.1: comentar brevemente los resultados obtenidos, y evaluar los cambios en las métricas de Fairness. Reflexionar sobre los beneficios y limitaciones


In [22]:
# de la sección 3 del taller formativo 2.2

def get_aif_metrics(attr, df, label_names):
    """
    Construye un BinaryLabelDataset de AIF360 a partir de un DataFrame
    y retorna las métricas de grupo junto con el dataset formateado.

    Parámetros
    ----------
    attr : str
        Nombre de la columna del atributo protegido (debe estar codificado como 0/1,
        donde 1 = grupo privilegiado).
    df : pd.DataFrame
        DataFrame que incluye features, atributo protegido y etiqueta.
    label_names : list[str]
        Lista con el nombre de la columna de etiqueta (p.ej. ["income"]).

    Retorna
    -------
    metric : BinaryLabelDatasetMetric
        Métricas de grupo sobre el dataset.
    dt : BinaryLabelDataset
        Dataset en formato AIF360.
    """
    privileged_groups   = [{attr: 1}]
    unprivileged_groups = [{attr: 0}]

    attributes_params = dict(protected_attribute_names=[attr], label_names=label_names)
    dt = BinaryLabelDataset(
        df=df, **attributes_params, favorable_label=1, unfavorable_label=0
    )

    metric = BinaryLabelDatasetMetric(
        dt, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups
    )

    return metric, dt

Pasamos primero las variables binarias del
df_test de la Parte 3 a valores numéricos:

In [23]:
# creamos las columnas numéricas en df_test
df_test["r3_num"] = df_test["r3_bin"].map({"privilegiado": 1, "no_privilegiado": 0}).astype("int64")
df_test["h7e_num"] = df_test["h7e_bin"].map({"privilegiado": 1, "no_privilegiado": 0}).astype("int64")

Luego creamos los dataframes numéricos:

In [24]:
# df con las etiquetas reales
df_real = pd.DataFrame({
    "r3_num": df_test["r3_num"],
    "h7e_num": df_test["h7e_num"],
    "pobreza": df_test["label_value"].astype("int64")
})

# df con las predicciones del modelo
df_pred = pd.DataFrame({
    "r3_num": df_test["r3_num"],
    "h7e_num": df_test["h7e_num"],
    "pobreza": df_test["score"].astype("int64")
})

Verificamos que todo sea numérico:

In [25]:
print("--- Tipos de df_real ---")
print(df_real.dtypes)
print("\n--- Tipos de df_pred ---")
print(df_pred.dtypes)


--- Tipos de df_real ---
r3_num     int64
h7e_num    int64
pobreza    int64
dtype: object

--- Tipos de df_pred ---
r3_num     int64
h7e_num    int64
pobreza    int64
dtype: object


In [26]:
# también de la sección 3 del taller formativo 2.2

def aplicar_EqOddsPostProcessing(group, df_pred, df_real):
    """
    Aplica EqOddsPostprocessing de AIF360 sobre las predicciones de un modelo
    para un atributo protegido dado, e imprime las métricas de fairness resultantes.

    Parámetros
    ----------
    group : str
        Nombre del atributo protegido (columna en df_test_pred y df_test,
        codificado como 0/1 donde 1 = privilegiado).
    df_test_pred : pd.DataFrame
        DataFrame del conjunto de test con las predicciones del modelo.
    df_test : pd.DataFrame
        DataFrame del conjunto de test con las etiquetas reales.
    """
    privileged_groups   = [{group: 1}]
    unprivileged_groups = [{group: 0}]

    bldm_antes, test_pred_aifdf = get_aif_metrics(group, df_pred, ["pobreza"])
    _, test_aifdf      = get_aif_metrics(group, df_real,      ["pobreza"])

    roc = EqOddsPostprocessing(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        seed=103,
    )

    cm_antes = ClassificationMetric(
        test_aifdf,
        test_pred_aifdf,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
    )

    roc.fit(test_aifdf, test_pred_aifdf)

    corrected_pred = roc.predict(test_pred_aifdf)

    bldm_despues = BinaryLabelDatasetMetric(
        corrected_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )

    cm = ClassificationMetric(
        test_aifdf,
        corrected_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
    )

    print(f"=== Atributo Protegido: {group} ===")
    print(f"  Disparate Impact (Antes)            : {bldm_antes.disparate_impact():.4f}")
    print(f"  Disparate Impact (Después)          : {bldm_despues.disparate_impact():.4f}")
    print(f"  Accuracy (Antes)                    : {cm_antes.accuracy():.4f}")
    print(f"  Accuracy (Después)                  : {cm.accuracy():.4f}")
    print(f"  Equal opportunity difference        : {cm.equal_opportunity_difference():.4f}")
    print(f"  Error rate difference               : {cm.error_rate_difference():.4f}")
    print(f"  FNR grupo privilegiado              : {cm.false_negative_rate(privileged=True):.4f}")
    print(f"  FNR grupo no privilegiado           : {cm.false_negative_rate(privileged=False):.4f}")
    print(f"  FNR ratio (no-priv / priv)          : {cm.false_negative_rate_ratio():.4f}")
    print(f"  FPR grupo privilegiado              : {cm.false_positive_rate(privileged=True):.4f}")
    print(f"  FPR grupo no privilegiado           : {cm.false_positive_rate(privileged=False):.4f}")
    print(f"  FPR ratio (no-priv / priv)          : {cm.false_positive_rate_ratio():.4f}")

Realizamos las llamadas de post procesamiento:

In [27]:
print("=== EqOddsPostprocessing: Grupos por Pertenencia Indígena (r3_num) ===")
aplicar_EqOddsPostProcessing("r3_num", df_pred, df_real)

print("\n=== EqOddsPostprocessing: Grupos por Dificultades de Autocuidado (h7e_num) ===")
aplicar_EqOddsPostProcessing("h7e_num", df_pred, df_real)

=== EqOddsPostprocessing: Grupos por Pertenencia Indígena (r3_num) ===
=== Atributo Protegido: r3_num ===
  Disparate Impact (Antes)            : 1.3140
  Disparate Impact (Después)          : 1.0759
  Accuracy (Antes)                    : 0.7424
  Accuracy (Después)                  : 0.6908
  Equal opportunity difference        : -0.0007
  Error rate difference               : -0.0073
  FNR grupo privilegiado              : 0.2134
  FNR grupo no privilegiado           : 0.2140
  FNR ratio (no-priv / priv)          : 1.0031
  FPR grupo privilegiado              : 0.3296
  FPR grupo no privilegiado           : 0.3301
  FPR ratio (no-priv / priv)          : 1.0013

=== EqOddsPostprocessing: Grupos por Dificultades de Autocuidado (h7e_num) ===
=== Atributo Protegido: h7e_num ===
  Disparate Impact (Antes)            : 1.2326
  Disparate Impact (Después)          : 1.0901
  Accuracy (Antes)                    : 0.7424
  Accuracy (Después)                  : 0.6779
  Equal opportunity diff

Al aplicar el algoritmo EqOddsPostprocessing vemos una mejora clara en las métricas de equidad para ambos atributos protegidos, en pertenencia indígena r3_num el disparate impact baja de 1.3140 a un valor bastante cercano al ideal de 1.0759, mientras que tanto el fnr ratio como el fpr ratio quedan prácticamente nivelados en 1.0031 y 1.0013 respectivamente, por el lado de dificultades de autocuidado h7e_num el disparate impact pasa de 1.2326 a 1.0901, y la disparidad previa en falsos negativos que rondaba 1.27 se corrige hasta alcanzar un fnr ratio de 0.9966, dejando tasas de error casi idénticas en torno al 28.6% para ambos grupos.

Entre las ventajas principales destaca que el método corrige de forma efectiva las diferencias sistemáticas en falsos positivos y falsos negativos entre grupos, además de ofrecer una solución post hoc que permite intervenir las predicciones de un modelo ya entrenado sin tocar los datos de origen ni asumir el costo de volver a entrenar.

El costo principal se nota en la exactitud global porque para forzar la equidad el algoritmo modifica predicciones válidas en ciertos casos, lo que reduce el accuracy de 0.7424 a 0.6908 en r3_num y de 0.7424 a 0.6779 en h7e_num, esto abre una discusión relevante en políticas públicas al clasificar de forma errónea a hogares que presentan carencias reales por efecto del ajuste, planteando la duda de si conviene priorizar la exactitud general promedio o asegurar que las tasas de error se repartan de manera uniforme entre todas las poblaciones.

## **Parte 5** (bonus +5 decimas) :  

Aplique el algoritmo de Reweighing sobre los datos de entrenamiento. Compare las métricas de Fairness para los datos de entrenamiento antes y después de aplicar Reweighing.

Luego, entrene un Arbol de decisión utilizando los datos de entrenamiento modificados y obtenga las métricas de Fairness de las predicciones de dicho modelo.

Analice cómo varió el *accuracy* del modelo respecto al original y las métricas de Fairness y reflexione sobre las implicancias sociales de estas decisiones.

### Indicaciones:
Puedes considerar los grupos priviligiados/no priviligiados mencionados en la parte 3. Considere al menos 2.

In [28]:
### codigo aquí

## Referencias y Declaración de IA

Siguiendo las pautas de integridad académica fijadas para este taller declaramos que recurrimos a la asistencia de inteligencia artificial exclusivamente en la parte 4, prescindiendo por completo de estas herramientas en el resto del trabajo, en dicha sección utilizamos Claude para comprender los conceptos de AIF360 y EqOddsPostprocessing en paralelo con el material del taller formativo 2.2, además de aclarar dudas y resolver errores vinculados a las estructuras de objetos que exige la librería y mejorar la presentación visual de las tablas y métricas de desempeño, la interacción se limitó a un esquema de preguntas orientadoras sin solicitarle la generación de código ni la redacción de respuestas, por lo que la implementación se basa en el taller formativo 2.2 y la elaboración de los textos es de nuestra autoría, previo a la entrega consultamos al equipo docente sobre la validez de este procedimiento y obtuvimos su confirmación, sumado a que los conceptos incluidos en la reflexión cuentan con sus respectivas referencias bibliográficas y el registro íntegro de la conversación se encuentra disponible a través del siguiente enlace: https://drive.google.com/drive/folders/1oqBhYN6de62FPsLSF4EBG7o8LP8gdFOF?usp=drive_link.

### **Referencias Bibliográficas**
- Ministerio de Desarrollo Social y Familia. (2024). *Metodología de diseño muestral Encuesta Casen 2024*. Subsecretaría de Evaluación Social, División Observatorio Social, Gobierno de Chile.
- Bellamy, R. K. et al. (2018). *AI Fairness 360: An extensible toolkit for detecting, understanding, and mitigating unwanted algorithmic bias*. arXiv preprint arXiv:1810.01909.